# Создание и заполнение данных БД Postgre

In [45]:
%pip install python-dotenv psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import os
import json
import psycopg2 # Основная библиотека для подключения к PostgreSQL из Python
from psycopg2.extras import DictCursor # DictCursor позволяет обращаться к колонкам результата по имени, а не по индексу
from dotenv import load_dotenv # Модуль для загрузки переменных окружения из файла .env


# Получение секретов

In [47]:
current_dir = os.getcwd()                     # os.getcwd() возвращает текущую рабочую директорию
project_root = os.path.dirname(current_dir)   # Поднимаемся на один уровень вверх – в корень проекта
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env') # Формируем полный путь к файлу .env, который находится в папке task_2_Docker
load_dotenv(dotenv_path) # Загружаем переменные из указанного .env-файла в переменные окружения процесса Python

user = os.getenv("USER")
password = os.getenv("PASSWORD")
db_name = os.getenv("DB")
db_port = os.getenv("DB_PORT")


print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Korchagina, DB=my_db_Korchagina, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [48]:
conn = None
try: # Устанавливаем соединение с PostgreSQL, используя параметры из .env
    conn = psycopg2.connect(
    host="localhost",
    database=db_name,
    user=user,
    password=password,
    port=db_port
    )
    cursor = conn.cursor() # Создаём курсор для выполнения SQL-запросов

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [49]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.16 (Debian 13.16-1.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit',)


In [ ]:
# получить список таблиц:
cursor.execute("""
SELECT table_name 
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall() # Получаем все строки результата
print("\nТаблицы в базе данных:") # Печатаем заголовок перед списком
for table in tables: # Перебираем каждую таблицу из полученного списка
    print(f"- {table[0]}") # Выводим имя таблицы (первый элемент кортежа)


Таблицы в базе данных:
- user_logs
- departments


Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [ ]:
import pandas as pd
cursor.execute("SELECT * FROM user_logs LIMIT 10;")
df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description]) # Заголовки колонок берём из описания курсора 
display(df)

,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt,depart
0,71262,34618,6,16,16.0,9,9.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
1,71262,34618,7,4,10.0,3,6.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
2,71262,34620,6,26,26.0,15,15.0,0,0.0,0,...,0.0,5,Экзамен,1,1,1,2,2,18.06.2022,22
3,71262,35444,6,5,5.0,1,1.0,0,0.0,0,...,0.0,5,Экзамен,1,1,1,2,2,18.06.2022,22
4,71262,35471,6,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
5,71262,35471,7,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
6,71262,35471,8,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
7,71262,35471,9,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
8,71262,35471,10,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22
9,71262,35471,11,0,0.0,0,0.0,0,0.0,0,...,0.0,4,Экзамен,1,1,1,2,2,18.06.2022,22


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [ ]:
cursor.execute("SELECT COUNT(DISTINCT depart) AS departments_count FROM user_logs;")
result = cursor.fetchone() # Забираем одну строку результата (кортеж с одним значением)
print(f"Количество кафедр: {result[0]}") # Выводим первое (и единственное) значение – число кафедр

Количество кафедр: 43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [ ]:
conn.rollback()
query_base = """
SELECT d.name AS department_name, COUNT(DISTINCT u.courseid) AS courses_count
FROM user_logs u
JOIN departments d ON u.depart = d.id
GROUP BY d.name
ORDER BY courses_count DESC
"""
cursor.execute(query_base)
all_depts = cursor.fetchall()
df = pd.DataFrame(all_depts, columns=['department_name', 'courses_count'])
display(df)

# Отдельный запрос на максимум
cursor.execute(query_base + " LIMIT 1") # Берём первую строку (самую большую courses_count)
max_dept = cursor.fetchone() # Получаем одну строку
print(f"Больше всего курсов у кафедры '{max_dept[0]}' — {max_dept[1]} курсов.")

,department_name,courses_count
0,ДиСО,53
1,ПОиД,42
2,МиХТ,42
3,ГМиТТК,40
4,ЛиУТС,36
5,БИиИТ,35
6,ГМУиУП,35
7,ПиЭММО,33
8,АЭПиМ,33
9,ЛиП,33


Больше всего курсов у кафедры 'ДиСО' — 53 курсов.


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [54]:
# Количество курсов, закреплённых за несколькими кафедрами
cursor.execute("""
SELECT COUNT(*) FROM (
    SELECT courseid FROM user_logs GROUP BY courseid HAVING COUNT(DISTINCT depart) > 1
) AS multi;
""")
count_multi = cursor.fetchone()[0]
print(f"Количество курсов, за которыми закреплено несколько кафедр: {count_multi}")

# Названия кафедр, совместно преподающих одни и те же курсы
cursor.execute("""
SELECT u.courseid, array_agg(DISTINCT d.name) AS departments
FROM user_logs u
JOIN departments d ON u.depart = d.id
GROUP BY u.courseid
HAVING COUNT(DISTINCT u.depart) > 1;
""")
rows = cursor.fetchall()
for row in rows:
    print(f"Курс {row[0]}: кафедры {', '.join(row[1])}")

Количество курсов, за которыми закреплено несколько кафедр: 60
Курс 71495: кафедры ГМиТТК, ПиСЗ, УиИС
Курс 71508: кафедры ПиСЗ, УиИС
Курс 71541: кафедры ПиСЗ, УиИС
Курс 71547: кафедры ПиСЗ, УиИС
Курс 71549: кафедры ГМиТТК, ТиЭС
Курс 71571: кафедры ЛиУТС, ПиСЗ, УиИС
Курс 71632: кафедры CC, ВТиП
Курс 71736: кафедры ГМДиОПИ, ЭиМЭ
Курс 71852: кафедры АЭПиМ, ЭПП
Курс 71857: кафедры АЭПиМ, ЭПП
Курс 71884: кафедры АЭПиМ, ЭПП
Курс 71892: кафедры АЭПиМ, ТиЭС, ЭПП
Курс 71904: кафедры АЭПиМ, ЭПП
Курс 72126: кафедры АЭПиМ, УиИС
Курс 72314: кафедры Дизайна, ЛПиМ
Курс 72347: кафедры Дизайна, ЛПиМ
Курс 72358: кафедры МиХТ, ТОМ
Курс 72359: кафедры ЛПиМ, МиХТ, ТОМ
Курс 72380: кафедры ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72392: кафедры ЛПиМ, МиХТ, ТОМ
Курс 72402: кафедры ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72416: кафедры ЛПиМ, МиХТ, ТОМ
Курс 72447: кафедры ЛПиМ, МиХТ, ТОМ
Курс 72457: кафедры ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72460: кафедры ЛПиМ, МиХТ, ТОМ
Курс 72462: кафедры МиХТ, ТОМ
Курс 72469: кафедры МиХТ, ТОМ
Курс 

## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [55]:
cursor.execute("""
SELECT namer_level, COUNT(DISTINCT userid) AS student_count
FROM user_logs
WHERE namer_level IN ('2', '3', '4', '5')
GROUP BY namer_level
ORDER BY namer_level;
""")
df = pd.DataFrame(cursor.fetchall(), columns=['namer_level', 'student_count'])
print(df.to_string(index=False))

namer_level  student_count
          2           1069
          3           1884
          4           3243
          5           3407


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [56]:
cursor.execute("""
SELECT userid, SUM(s_all) AS total_logs
FROM user_logs
GROUP BY userid
ORDER BY total_logs DESC
LIMIT 1;
""")
userid, total = cursor.fetchone()
print(f"Студент {userid} имеет максимальное количество логов: {total}")

Студент 28871 имеет максимальное количество логов: 10300


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [ ]:
# Получаем данные
cursor.execute("""
SELECT num_week, AVG(s_all) AS avg_events
FROM user_logs
GROUP BY num_week
ORDER BY num_week;
""")
df = pd.DataFrame(cursor.fetchall(), columns=['num_week', 'avg_events'])

df['avg_events'] = pd.to_numeric(df['avg_events']) # Приводим текстовые числа к float
df['avg_events'] = df['avg_events'].round(2) # Округляем до двух десятичных знаков

# Выводим без индекса
print(df.to_string(index=False))

 num_week  avg_events
        6       13.80
        7        9.62
        8        8.03
        9        9.39
       10        8.21
       11       10.02
       12        9.38
       13       10.01
       14        9.86
       15       10.35
       16       10.29
       17       10.52
       18        9.67
       19       11.11
       20       14.45
       21       18.50
       22       22.49
       23       22.26
       24       23.01
       25       18.22
       26        8.60
       27        1.25
       28        0.09
       29        0.05


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [58]:
# Отличники
cursor.execute("""
SELECT d.name, COUNT(DISTINCT u.userid) AS excellent_students
FROM user_logs u
JOIN departments d ON u.depart = d.id
WHERE u.namer_level = '5'
GROUP BY d.name
ORDER BY excellent_students DESC
LIMIT 1;
""")
dept, cnt = cursor.fetchone()
print(f"Кафедра с наибольшим количеством отличников: {dept} ({cnt} студентов)")

# Двоечники
cursor.execute("""
SELECT d.name, COUNT(DISTINCT u.userid) AS bad_students
FROM user_logs u
JOIN departments d ON u.depart = d.id
WHERE u.namer_level = '2'
GROUP BY d.name
ORDER BY bad_students DESC
LIMIT 1;
""")
dept, cnt = cursor.fetchone()
print(f"Кафедра с наибольшим количеством двоечников: {dept} ({cnt} студентов)")

Кафедра с наибольшим количеством отличников: ДиСО (310 студентов)
Кафедра с наибольшим количеством двоечников: Эконом. (72 студентов)


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [59]:
query = """
WITH weekly_activity AS (
    SELECT 
        CASE 
            WHEN name_osno = '1' THEN 'бюджет'
            WHEN name_osno = '2' THEN 'контракт'
            ELSE name_osno
        END AS name_osno_text,
        num_week,
        AVG(s_all) AS avg_s_all,
        AVG(s_course_viewed) AS avg_s_course_viewed,
        AVG(s_q_attempt_viewed) AS avg_s_q_attempt_viewed
    FROM user_logs
    GROUP BY name_osno, num_week
),
max_week AS (
    SELECT num_week
    FROM weekly_activity
    GROUP BY num_week
    ORDER BY AVG(avg_s_all) DESC
    LIMIT 1
)
SELECT 
    wa.name_osno_text AS name_osno,
    wa.num_week AS week_number,
    wa.avg_s_all,
    wa.avg_s_course_viewed,
    wa.avg_s_q_attempt_viewed
FROM weekly_activity wa
JOIN max_week mw ON wa.num_week = mw.num_week
ORDER BY wa.name_osno_text;
"""
cursor.execute(query)
df = pd.DataFrame(cursor.fetchall(), columns=['name_osno', 'week_number', 
'avg_s_all', 'avg_s_course_viewed', 'avg_s_q_attempt_viewed'])

# Приведение к числовым типам и округление
for col in ['avg_s_all', 'avg_s_course_viewed', 'avg_s_q_attempt_viewed']:
    df[col] = pd.to_numeric(df[col]).round(2)

# Вывод без индекса
print(df.to_string(index=False))

name_osno  week_number  avg_s_all  avg_s_course_viewed  avg_s_q_attempt_viewed
   бюджет           24      20.79                 3.78                    4.94
 контракт           24      28.69                 4.73                    5.42


In [60]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()